# 0. Library and settings

In [ ]:
# Library
import os
import pickle
import math
import numpy as np
import pandas as pd

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, Range1d
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.io import export_png

In [ ]:
# Make directories
os.makedirs('Fig', exist_ok=True)

In [ ]:
# Figure number definitions
Fig1d = f'Fig/Fig1(d)'
Supplementary_Fig2 = f'Fig/Supplementary_Fig2'

In [ ]:
# Functions
def rgb_to_hex(rgb):
    return '#' + '{:X}'.format(rgb[0]).zfill(2) + '{:X}'.format(rgb[1]).zfill(2) + '{:X}'.format(rgb[2]).zfill(2)

# 1. Complex Heatmap (GEX)

## 1.1. Get top 75 marker genes

In [ ]:
################################
# Cluster definition file for heatmap
################################
# Loading cluster definition
df_cdef = pd.read_csv(f'cluster_def_by_monocle_final.v2.1.rev.csv', index_col=0)

# Rename cluster numbers
strings = ['a' if b[-1] == '1' else 'b' if b[-1] == '2' else 'c' for b in df_cdef.index.tolist()]
cnums = df_cdef['monocle_clusters'].tolist()
df_cdef['monocle_clusters'] = [f'{cn}{s}'for cn, s in zip(cnums, strings)]

# Rename barcodes
df_cdef.index = [f'pln_{b[:-2]}' if b[-1] == '1' else f'mln_{b[:-2]}' if b[-1] == '2' else f'pp_{b[:-2]}'for b in df_cdef.index.tolist()]

# Save
df_cdef.to_csv(f'cluster_def_by_monocle_final.v2.1.rev.heatmap.csv')

In [ ]:
################################
# Update GEX/ATAC object
################################
rscript1 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: update_gex_atac_abject.r
# --------------------------------------------------------
# How to run
# $ Rscript update_gex_atac_abject.r
# --------------------------------------------------------
# outputs
# - Seurat/PLN/out/scaled_data.csv
# - Seurat/MLN/out/scaled_data.csv
# - Seurat/PP/out/scaled_data.csv
# - Seurat/out/scaled_data.csv
# - Seurat/PLN/out/gex.atac.v2.fixed.RData
# - Seurat/MLN/out/gex.atac.v2.fixed.RData
# - Seurat/PP/out/gex.atac.v2.fixed.RData
# - Seurat/out/gex.atac.v2.fixed.RData
################################
# -----------------------------------------------------------------------------------------
# 1. Preparation for updating GEX/ATAC objects: Monocle3
# -----------------------------------------------------------------------------------------
# Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(SeuratWrappers)
library(stringr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(patchwork)
set.seed(1234)

# Memory
options(future.globals.maxSize = 16000 * 1024^2)

# Load GEX data
cds1 <- load_mm_data(
    mat_path = "PLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "PLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

cds2 <- load_mm_data(
    mat_path = "MLN/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "MLN/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "MLN/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

cds3 <- load_mm_data(
    mat_path = "PP/filtered_feature_bc_matrix/matrix_gex.v2.rev.mtx", 
    feature_anno_path = "PP/filtered_feature_bc_matrix/features_gex.v2.rev.tsv", 
    cell_anno_path = "PP/filtered_feature_bc_matrix/barcodes_gex.v2.rev.tsv"
)

# Rename
colnames(cds1) <- str_c("pln_", colnames(cds1), sep="")
colnames(cds2) <- str_c("mln_", colnames(cds2), sep="")
colnames(cds3) <- str_c("pp_", colnames(cds3), sep="")

# Cell selection
load("Signac/atac.v2.RData")
cds1 <- cds1[, colnames(cds1) %in% rownames(combined@meta.data)]
cds2 <- cds2[, colnames(cds2) %in% rownames(combined@meta.data)]
cds3 <- cds3[, colnames(cds3) %in% rownames(combined@meta.data)]

# Remove batch effect
cds <- combine_cds(list(cds1, cds2, cds3))
cds <- preprocess_cds(cds, num_dim = 20) # =PCA
cds <- align_cds(cds, num_dim = 100, alignment_group = "sample") # =Correct PCA

# Rename (ALL)
colnames(cds) <- str_replace(colnames(cds), pattern="_1", replacement="")
colnames(cds) <- str_replace(colnames(cds), pattern="_2", replacement="")
colnames(cds) <- str_replace(colnames(cds), pattern="_3", replacement="")

# Re-rename
colnames(cds1) <- str_replace(colnames(cds1), pattern="pln_", replacement="")
colnames(cds2) <- str_replace(colnames(cds2), pattern="mln_", replacement="")
colnames(cds3) <- str_replace(colnames(cds3), pattern="pp_", replacement="")

# Gene short name
names(rowData(cds1)) <- "gene_short_name"
names(rowData(cds2)) <- "gene_short_name"
names(rowData(cds3)) <- "gene_short_name"
names(rowData(cds)) <- "gene_short_name"
rownames(cds1) <- rowData(cds1)[rownames(cds1),]$gene_short_name
rownames(cds2) <- rowData(cds2)[rownames(cds2),]$gene_short_name
rownames(cds3) <- rowData(cds3)[rownames(cds3),]$gene_short_name
rownames(cds) <- rowData(cds)[rownames(cds),]$gene_short_name

# Convert to Seurat object
cds1.seurat <- as.Seurat(cds1, assay = NULL)
cds2.seurat <- as.Seurat(cds2, assay = NULL)
cds3.seurat <- as.Seurat(cds3, assay = NULL)
cds.seurat <- as.Seurat(cds, assay = NULL)


# -----------------------------------------------------------------------------------------
# 2. Preparation for updating GEX/ATAC objects: Seurat
# -----------------------------------------------------------------------------------------
# load cluster definition
cdef1 <- read.csv("cluster_def_by_monocle_final.v2.1.rev.heatmap.csv", row.names=1)

# add cluster definitions
cds1.seurat@meta.data["cdef1"] <- cdef1.pln[rownames(cds1.seurat@meta.data),]
cds2.seurat@meta.data["cdef1"] <- cdef1.mln[rownames(cds2.seurat@meta.data),]
cds3.seurat@meta.data["cdef1"] <- cdef1.pp[rownames(cds3.seurat@meta.data),]
cds.seurat@meta.data["cdef1"] <- cdef1[rownames(cds.seurat@meta.data),]

# calc some features required for DoHeatMap
cds1.seurat <- NormalizeData(object=cds1.seurat, normalization.method="LogNormalize", scale.factor = 10000)
cds1.seurat <- FindVariableFeatures(object=cds1.seurat, nfeatures = 2000)
cds1.seurat <- ScaleData(object=cds1.seurat)
cds2.seurat <- NormalizeData(object=cds2.seurat, normalization.method="LogNormalize", scale.factor = 10000)
cds2.seurat <- FindVariableFeatures(object=cds2.seurat, nfeatures = 2000)
cds2.seurat <- ScaleData(object=cds2.seurat)
cds3.seurat <- NormalizeData(object=cds3.seurat, normalization.method="LogNormalize", scale.factor = 10000)
cds3.seurat <- FindVariableFeatures(object=cds3.seurat, nfeatures = 2000)
cds3.seurat <- ScaleData(object=cds3.seurat)
cds.seurat <- NormalizeData(object=cds.seurat, normalization.method="LogNormalize", scale.factor = 10000)
cds.seurat <- FindVariableFeatures(object=cds.seurat, nfeatures = 2000)
cds.seurat <- ScaleData(object=cds.seurat)

# save scaled values
write.csv(
    cds1.seurat[["originalexp"]]@scale.data,
    file = "Seurat/PLN/out/scaled_data.csv",
    quote = FALSE,
    row.names = TRUE
)
write.csv(
    cds2.seurat[["originalexp"]]@scale.data,
    file = "Seurat/MLN/out/scaled_data.csv",
    quote = FALSE,
    row.names = TRUE
)
write.csv(
    cds3.seurat[["originalexp"]]@scale.data,
    file = "Seurat/PP/out/scaled_data.csv",
    quote = FALSE,
    row.names = TRUE
)
write.csv(
    cds.seurat[["originalexp"]]@scale.data,
    file = "Seurat/out/scaled_data.csv",
    quote = FALSE,
    row.names = TRUE
)


# -----------------------------------------------------------------------------------------
# 3. Add ATAC/WNN data
# -----------------------------------------------------------------------------------------
# load ATAC data
load("Signac/PLN/pln.atac.v2.RData")
load("Signac/MLN/mln.atac.v2.RData")
load("Signac/PP/pp.atac.v2.RData")

# add ATAC
cds1.seurat[["ATAC"]] <- pln[["peaks"]]
cds2.seurat[["ATAC"]] <- mln[["peaks"]]
cds3.seurat[["ATAC"]] <- pp[["peaks"]]

# RNA analysis
cds1.seurat <- SCTransform(cds1.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')
cds2.seurat <- SCTransform(cds2.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')
cds3.seurat <- SCTransform(cds3.seurat, verbose = FALSE, assay="originalexp") %>% RunPCA() %>% RunUMAP(dims = 1:50, reduction.name = 'umap.rna', reduction.key = 'rnaUMAP_')

# ATAC analysis
DefaultAssay(cds1.seurat) <- "ATAC"
cds1.seurat <- RunTFIDF(cds1.seurat)
cds1.seurat <- FindTopFeatures(cds1.seurat, min.cutoff = 'q0')
cds1.seurat <- RunSVD(cds1.seurat)
cds1.seurat <- RunUMAP(cds1.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")
DefaultAssay(cds2.seurat) <- "ATAC"
cds2.seurat <- RunTFIDF(cds2.seurat)
cds2.seurat <- FindTopFeatures(cds2.seurat, min.cutoff = 'q0')
cds2.seurat <- RunSVD(cds2.seurat)
cds2.seurat <- RunUMAP(cds2.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")
DefaultAssay(cds3.seurat) <- "ATAC"
cds3.seurat <- RunTFIDF(cds3.seurat)
cds3.seurat <- FindTopFeatures(cds3.seurat, min.cutoff = 'q0')
cds3.seurat <- RunSVD(cds3.seurat)
cds3.seurat <- RunUMAP(cds3.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")

# WNN
cds1.seurat <- FindMultiModalNeighbors(cds1.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds1.seurat <- RunUMAP(cds1.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
cds2.seurat <- FindMultiModalNeighbors(cds2.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds2.seurat <- RunUMAP(cds2.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
cds3.seurat <- FindMultiModalNeighbors(cds3.seurat, reduction.list = list("pca", "lsi"), dims.list = list(1:50, 2:50))
cds3.seurat <- RunUMAP(cds3.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")

# save
save(cds1.seurat, file="Seurat/PLN/out/gex.atac.v2.fixed.RData")
save(cds2.seurat, file="Seurat/MLN/out/gex.atac.v2.fixed.RData")
save(cds3.seurat, file="Seurat/PP/out/gex.atac.v2.fixed.RData")

# For combined data
cds.seurat[["ATAC"]] <- combined[["peaks"]]
# Reduce dimension (GEX)
cds.seurat <- RunUMAP(object=cds.seurat, reduction='Aligned', dims=1:20, reduction.name = 'umap.aligned', reduction.key = 'alignedUMAP_')
# Reduce dimension (ATAC)
DefaultAssay(cds.seurat) <- "ATAC"
cds.seurat <- RunTFIDF(cds.seurat)
cds.seurat <- FindTopFeatures(cds.seurat, min.cutoff = 20)
cds.seurat <- RunSVD(cds.seurat)
cds.seurat <- RunUMAP(cds.seurat, reduction = 'lsi', dims = 2:50, reduction.name = "umap.atac", reduction.key = "atacUMAP_")
# WNN
cds.seurat <- FindMultiModalNeighbors(cds.seurat, reduction.list = list("Aligned", "lsi"), dims.list = list(1:20, 2:50))
cds.seurat <- RunUMAP(cds.seurat, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
# save
save(cds.seurat, file="Seurat/out/gex.atac.v2.fixed.RData")
"""

f = open(f'update_gex_atac_abject.r', 'w')
f.write(rscript1)
f.close()

In [ ]:
!Rscript update_gex_atac_abject.r

In [ ]:
################################
# Make maker gene list
################################
rscript2 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: make_maker_gene_list.r
# --------------------------------------------------------
# How to run
# $ Rscript make_maker_gene_list.r
# --------------------------------------------------------
# outputs
# - Seurat/PLN/out/markers.gex.v2.1.RData
# - Seurat/PLN/out/markers.gex.top75.v2.1.csv
# - Seurat/PLN/out/markers.gex.heatmap.v2.1.pdf
# - Seurat/PLN/out/markers.gex.pathway.v2.1.pdf
# - Seurat/PLN/out/ridge_plot.v2.1.pdf
# - Seurat/PLN/out/violin_plot.v2.1.pdf
# - Seurat/PLN/out/dot_plot.v2.1.pdf
# - Seurat/MLN/out/markers.gex.v2.1.RData
# - Seurat/MLN/out/markers.gex.top75.v2.1.csv
# - Seurat/MLN/out/markers.gex.heatmap.v2.1.pdf
# - Seurat/MLN/out/markers.gex.pathway.v2.1.pdf
# - Seurat/MLN/out/ridge_plot.v2.1.pdf
# - Seurat/MLN/out/violin_plot.v2.1.pdf
# - Seurat/MLN/out/dot_plot.v2.1.pdf
# - Seurat/PP/out/markers.gex.v2.1.RData
# - Seurat/PP/out/markers.gex.top75.v2.1.csv
# - Seurat/PP/out/markers.gex.heatmap.v2.1.pdf
# - Seurat/PP/out/markers.gex.pathway.v2.1.pdf
# - Seurat/PP/out/ridge_plot.v2.1.pdf
# - Seurat/PP/out/violin_plot.v2.1.pdf
# - Seurat/PP/out/dot_plot.v2.1.pdf
# - Seurat/out/markers.gex.v2.1.RData
# - Seurat/out/markers.gex.top75.v2.1.csv
# - Seurat/out/markers.gex.heatmap.v2.1.pdf
# - Seurat/out/markers.gex.pathway.v2.1.pdf
# - Seurat/out/ridge_plot.v2.1.pdf
# - Seurat/out/violin_plot.v2.1.pdf
# - Seurat/out/dot_plot.v2.1.pdf
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(SeuratWrappers)
library(stringr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(patchwork)
set.seed(1234)

# Memory
options(future.globals.maxSize = 16000 * 1024^2)

# Function
markerGenes <- function(obj, tissue="PLN"){{
        # swithcing
        Idents(obj) <- "cdef1"

        # rename clusters
        obj <- RenameIdents(obj, '0' = 'Unknown')
        obj <- RenameIdents(obj, '1a' = 'Art')
        obj <- RenameIdents(obj, '3a' = "CapEC2'")
        obj <- RenameIdents(obj, '4a' = 'CapEC2')
        obj <- RenameIdents(obj, '5a' = 'CapEC1')
        obj <- RenameIdents(obj, '6a' = 'CRP')
        obj <- RenameIdents(obj, '7a' = 'TrEC')
        obj <- RenameIdents(obj, '8a' = 'HEC')
        obj <- RenameIdents(obj, '9a' = 'Vn')
        obj <- RenameIdents(obj, '1b' = 'Art')
        obj <- RenameIdents(obj, '3b' = "CapEC2'")
        obj <- RenameIdents(obj, '4b' = 'CapEC2')
        obj <- RenameIdents(obj, '5b' = 'CapEC1')
        obj <- RenameIdents(obj, '6b' = 'CRP')
        obj <- RenameIdents(obj, '7b' = 'TrEC')
        obj <- RenameIdents(obj, '8b' = 'HEC')
        obj <- RenameIdents(obj, '9b' = 'Vn')
        obj <- RenameIdents(obj, '1c' = 'Art')
        obj <- RenameIdents(obj, '3c' = "CapEC2'")
        obj <- RenameIdents(obj, '4c' = 'CapEC2')
        obj <- RenameIdents(obj, '5c' = 'CapEC1')
        obj <- RenameIdents(obj, '6c' = 'CRP')
        obj <- RenameIdents(obj, '7c' = 'TrEC')
        obj <- RenameIdents(obj, '8c' = 'HEC')
        obj <- RenameIdents(obj, '9c' = 'Vn')

        # set plotting order
        levels(obj) <- c(
                "Art",
                "CapEC2'",
                "CapEC2",
                "CapEC1",
                "CRP",
                "TrEC",
                "HEC",
                "Vn"
        )

        # set color
        colors = c(
                "#D0021B",
                "#BA98FF",
                "#FF789B",
                "#50E3C2",
                "#7ED321",
                "#4A90E2",
                "#F5A623",
                "#9013FE",
        )

        # path
        if (tissue == "PLN"){{
                path_marker_obj = "Seurat/PLN/out/markers.gex.v2.1.RData"
                path_top75 = "Seurat/PLN/out/markers.gex.top75.v2.1.csv"
                path_heatmap = "Seurat/PLN/out/markers.gex.heatmap.v2.1.pdf"
                path_pathway = "Seurat/PLN/out/markers.gex.pathway.v2.1.pdf"
                path_ridge = "Seurat/PLN/out/ridge_plot.v2.1.pdf"
                path_violin = "Seurat/PLN/out/violin_plot.v2.1.pdf"
                path_dot = "Seurat/PLN/out/dot_plot.v2.1.pdf"
        }} else if (tissue == "MLN"){{
                path_marker_obj = "Seurat/MLN/out/markers.gex.v2.1.RData"
                path_top75 = "Seurat/MLN/out/markers.gex.top75.v2.1.csv"
                path_heatmap = "Seurat/MLN/out/markers.gex.heatmap.v2.1.pdf"
                path_pathway = "Seurat/MLN/out/markers.gex.pathway.v2.1.pdf"
                path_ridge = "Seurat/MLN/out/ridge_plot.v2.1.pdf"
                path_violin = "Seurat/MLN/out/violin_plot.v2.1.pdf"
                path_dot = "Seurat/MLN/out/dot_plot.v2.1.pdf"
        }} else if (tissue == "PP"){{
                path_marker_obj = "Seurat/PP/out/markers.gex.v2.1.RData"
                path_top75 = "Seurat/PP/out/markers.gex.top75.v2.1.csv"
                path_heatmap = "Seurat/PP/out/markers.gex.heatmap.v2.1.pdf"
                path_pathway = "Seurat/PP/out/markers.gex.pathway.v2.1.pdf"
                path_ridge = "Seurat/PP/out/ridge_plot.v2.1.pdf"
                path_violin = "Seurat/PP/out/violin_plot.v2.1.pdf"
                path_dot = "Seurat/PP/out/dot_plot.v2.1.pdf"
        }} else {{
                path_marker_obj = "Seurat/out/markers.gex.v2.1.RData"
                path_top75 = "Seurat/out/markers.gex.top75.v2.1.csv"
                path_heatmap = "Seurat/out/markers.gex.heatmap.v2.1.pdf"
                path_pathway = "Seurat/out/markers.gex.pathway.v2.1.pdf"
                path_ridge = "Seurat/out/ridge_plot.v2.1.pdf"
                path_violin = "Seurat/out/violin_plot.v2.1.pdf"
                path_dot = "Seurat/out/dot_plot.v2.1.pdf"
        }}

        # marker genes
        DefaultAssay(obj) <- "originalexp"
        obj.markers.gex <- FindAllMarkers(
                object = obj,
                only.pos = FALSE,
                min.pct = 0.25,
                logfc.threshold = 0.25,
                test.use = "MAST"
        )

        # save
        save(obj.markers.gex, file=path_marker_obj)
        obj.markers.gex %>%
                dplyr::filter(avg_log2FC >= 0) %>%
                group_by(cluster) %>%
                top_n(75, avg_log2FC) -> top75.posi
        write.csv(
                top75.posi,
                file = path_top75,
                quote = FALSE,
                row.names = FALSE
        )

        # heatmap
        p <- DoHeatmap(
                obj,
                features = top75.posi$gene,
                group.colors = colors,
                label = FALSE
        )
        ggsave(filename = path_heatmap, plot = p)

        # pathway analysis
        ck <- clusterEnrich(diff_genes = obj.markers.gex, diff_type="positive")
        p <- dotplot(ck, font.size = 10, showCategory=5)
        ggsave(filename = path_pathway, plot = p)

        # Ridge plots
        levels(obj) <- rev(levels(obj))
        p <- RidgePlot(
                obj,
                features = c("Glycam1", "Madcam1", "Esrrg", "Chst4"),
                ncol = 2,
                cols = rev(colors),
                slot = "scale.data"
        )
        ggsave(filename = path_ridge, plot = p)

        # violin plot
        levels(obj) <- rev(levels(obj))
        p <- VlnPlot(
                obj,
                features = c("Glycam1", "Madcam1", "Esrrg", "Chst4"),
                ncol = 2,
                cols = colors,
                slot = "scale.data"
        )
        ggsave(filename = path_violin, plot = p)

        # dot plot
        obj.markers.gex %>%
                dplyr::filter(avg_log2FC >= 0) %>%
                group_by(cluster) %>%
                top_n(3, avg_log2FC) -> top3.posi
        p <- DotPlot(obj, features = top3.posi$gene) + RotatedAxis()
        ggsave(filename = path_dot, plot = p)
}}

load("Seurat/PLN/out/gex.atac.v2.fixed.RData")
load("Seurat/MLN/out/gex.atac.v2.fixed.RData")
load("Seurat/PP/out/gex.atac.v2.fixed.RData")
load("Seurat/out/gex.atac.v2.fixed.RData")

obj <- list(cds1.seurat, cds2.seurat, cds3.seurat, cds.seurat)
tissue <- c("PLN", "MLN", "PP", "ALL")
for (t in 1:4) {{
    markerGenes(obj[[t]], tissue=tissue[t])
}}
"""

f = open(f'make_maker_gene_list.r', 'w')
f.write(rscript2)
f.close()

In [ ]:
!Rscript make_maker_gene_list.r

## 1.2. Draw heatmap

In [ ]:
def plot_complex_heatmap_all(tissue, version, genes, ratio, filename):
    # load csv data
    path = "Seurat/" + tissue + "/out/heatmap75.v" + version + ".csv"
    path = path.replace("/ALL", "")
    df = pd.read_csv(path, index_col=0)
    
    # load cluster definition
    with open("cluster_def_by_monocle_final.v" + version + ".pkl", "rb") as tf:
        cdef = pickle.load(tf)

    # tissue idxes
    idx = 1 if tissue == "PLN" else 2 if tissue == "MLN" else 3 if tissue == "PP" else 4    
    
    # cluster numbers
    nums = sorted(list(set(cdef.values())))
    
    # cluster color map
    # definition
    color_map = {
        1:"#D0021B",
        4:"#BA98FF",
        5:"#FF789B",
        6:"#50E3C2",
        7:"#7ED321",
        8:"#4A90E2",
        9:"#F5A623",
        10:"#9013FE",
        -1: "black",
    }
    legend_map = {
        1:"Art", # Art
        4:"CapEC2'", # CapEC2'
        5:"CapEC2", # CapEC2
        6:"CapEC1", # CapEC1
        7:"CRP", # CRP
        8:"TrEC", # TrEC
        9:"HEC", # HEC
        10:"Vn", # Vn
        -1:"Others", # Others
    }
        
    # tissue color map
    tissue_color_map = [
        "#1EB100",
        "#FEF156",
        "#ED220D"
    ]

    # tissue legend map
    tissue_legend_map = [
        "PLN",
        "MLN",
        "PP"
    ]

    # column rename
    cols = []
    for c in df.columns.tolist():
        if c.startswith("pln_"):
            cols.append(c.split("_")[1]+"_1")
        elif c.startswith("mln_"):
            cols.append(c.split("_")[1]+"_2")
        else:
            cols.append(c.split("_")[1]+"_3")

            
    # column reorder, downsampling and split info.
    cols_ = []
    colbar = []
    subcolbar = []
    for n in nums:
        subset = [i for i in cols if cdef[i] == n]
        subset_1 = [i for i in subset if i[-1] == "1"]
        subset_2 = [i for i in subset if i[-1] == "2"]
        subset_3 = [i for i in subset if i[-1] == "3"]
        cols_ += subset_1 + subset_2 + subset_3
        
    cols_ = [cols_[i] for i in range(len(cols_)) if i % ratio == 0]
    
    for n in nums:
        subset = [i for i in cols_ if cdef[i] == n]
        subset_1 = [i for i in subset if i[-1] == "1"]
        subset_2 = [i for i in subset if i[-1] == "2"]
        subset_3 = [i for i in subset if i[-1] == "3"]
        colbar.append(len(subset))
        subcolbar.append([len(subset_1), len(subset_2), len(subset_3)])

    cols_ = ["pln_"+i.split("_")[0] if i[-1]=="1" else "mln_"+i.split("_")[0] if i[-1]=="2" else "pp_"+i.split("_")[0] for i in cols_]
    df = df.loc[:, cols_]
            
    
    # row split
    rowbar = [100] * (len(nums))
    
    # custom color scale
    v = list(range(1, 256))[::-1]
    p_to_b = [(i, 0, i) for i in v]
    v = list(range(1, 256))
    b_to_y = [(i, i, 0) for i in v]
    custom_palette = [rgb_to_hex(i) for i in p_to_b + [(0, 0, 0)] + b_to_y]
    

    #################################
    # Parameters
    #################################
    f_width = 1500
    f_height = 900
    h_width = 1000
    h_height = 700
    s_width = 50
    s_height = 100
    e_width = 200
    e_height = 100
    w = h_width / len(df.T)
    h = h_height / len(df)
    q = 1
    
    col_split = q * 4
    row_split = 4
    
    color_max = 2.2
    color_min = -2.2 
    color_bar_x = s_width + h_width + q * 200
    color_bar_y = e_height - row_split * len(rowbar)
    color_bar_w = q * 35
    color_bar_h = 175
    color_bar_ticks_ratio = 6
    color_bar_ticks_line_width_ratio = 75
    color_bar_texts_font_size_ratio = 12
    
    rowbar_x = s_width - q * 17.5
    rowbar_w = q * 25
    
    colbar_y = e_height + h_height + 42.5
    colbar_h = 25

    subcolbar_y = e_height + h_height + 17.5
    subcolbar_h = 25

    ga_font_size = 22
    ga_line_length = 15
    ga_line_width = 2
    ga_factor = 1.2
    
    cluster_title_font_size = 25
    cluster_legend_font_size = 22
    cluster_legend_x = s_width + h_width + q * 200
    cluster_legend_y = e_height + h_height + 50

    tissue_title_font_size = 25
    tissue_legend_font_size = 22
    tissue_legend_x = s_width + h_width + q * 200
    tissue_legend_y = e_height + h_height + 50 - 462
    
    
    #################################
    # Figure
    #################################
    p = figure(tools="save", width=f_width, height=f_height)
    p.grid.grid_line_color = None
    p.axis.axis_line_color = None
    p.axis.major_tick_line_color = None
    p.x_range = Range1d(0, f_width)
    p.y_range = Range1d(0, f_height)
    

    #################################
    # Heatmap
    #################################
    x = []
    y = []
    val = []
    for i in range(len(df.T)):
        for j in range(len(df)):
            cs = col_split * len([k for k in range(len(colbar)) if i >= sum(colbar[:(k+1)])]) # column split
            rs = row_split * len([k for k in range(len(rowbar)) if j >= sum(rowbar[:(k+1)])]) # row split
            x.append(s_width + w * i + cs + w / 2)
            y.append(h_height + e_height - h * (j + 1) - rs + h / 2)    
            val.append(df.iloc[j, i])

    source = ColumnDataSource(dict(x=x, y=y, val=val))
    linear_transform_color = linear_cmap(field_name="val", palette=custom_palette, low=color_min, high=color_max)
    p.rect('x', 'y', w, h, source=source, color=linear_transform_color, line_color=linear_transform_color, line_width=1)


    #################################
    # Color bar
    #################################
    # color rects
    x = []
    y = []
    col = []
    hu = color_bar_h / len(custom_palette)
    for i in range(len(custom_palette)):
        x.append(color_bar_x)
        y.append(color_bar_y + i * hu + hu / 2)
        col.append(custom_palette[i])

    source = ColumnDataSource(dict(x=x, y=y, col=col))
    p.rect('x', 'y', color_bar_w, hu, source=source, color='col', line_color='col')
    
    # ticks
    high = math.floor(color_max)
    low = math.ceil(color_min)
    step = (high - low) // 5 if high - low >= 5 else 1
    steps = list(range(low, high + 1, step))
    xrs = color_bar_x + (color_bar_w / 2) - (color_bar_w / color_bar_ticks_ratio)
    xre = color_bar_x + (color_bar_w / 2)
    xls = color_bar_x - (color_bar_w / 2) + (color_bar_w / color_bar_ticks_ratio)
    xle = color_bar_x - (color_bar_w / 2)
    y = [color_bar_y + (i - color_min) *(color_bar_h / (color_max - color_min)) for i in steps]
    lw = color_bar_h / color_bar_ticks_line_width_ratio
    for y_ in y:
        p.line([xrs, xre], [y_, y_], line_color = "white", line_width = lw)
        p.line([xls, xle], [y_, y_], line_color = "white", line_width = lw)

    # texts
    x = [color_bar_x + (color_bar_w / 2) + (color_bar_w / color_bar_ticks_ratio)] * len(y)
    t = [str(s) for s in steps]
    source = ColumnDataSource(dict(x=x, y=y, t=t))
    font_size = str(round(color_bar_h / color_bar_texts_font_size_ratio))+"px"
    p.text(x='x', y='y', text='t', text_font_size=font_size, text_align="left", text_baseline="middle", source=source)
    
    
    #################################
    # row colors
    #################################
    y = [h_height + e_height - row_split * i - h * sum(rowbar[:i]) - h * (rowbar[i] / 2) for i in range(len(rowbar))]
    hs = [h * i for i in rowbar]
    c = [color_map[nums[i]] for i in range(len(rowbar))]
    source = ColumnDataSource(dict(y=y, h=hs, c=c))
    p.rect(rowbar_x, 'y', rowbar_w, 'h', color='c', line_color='c', source=source)

    
    #################################
    # col colors
    #################################
    x = [s_width + col_split * i + w * sum(colbar[:i]) + w * (colbar[i] / 2) for i in range(len(colbar))]
    ws = [w * i for i in colbar]
    c = [color_map[nums[i]] for i in range(len(colbar))]
    source = ColumnDataSource(dict(x=x, w=ws, c=c))
    p.rect('x' , colbar_y, 'w', colbar_h, color='c', line_color='c', source=source)


    #################################
    # subcol colors
    #################################
    x = []
    ws = []
    c = []
    for i in range(len(subcolbar)):
        for j in range(len(subcolbar[i])):
            x.append(s_width + col_split * i + w * sum(colbar[:i]) + w * sum(subcolbar[i][:j]) + w * (subcolbar[i][j] / 2))
            ws.append(w * subcolbar[i][j])
            c.append(tissue_color_map[j])
    source = ColumnDataSource(dict(x=x, w=ws, c=c))
    p.rect('x' , subcolbar_y, 'w', subcolbar_h, color='c', line_color='c', source=source)
    
    
    #################################
    # gene annotations
    #################################    
    genes_all = df.index.tolist()
    genes = [g for g in genes if g in genes_all]
    idxes = [[i for i in range(len(genes_all)) if genes_all[i] == g][0] for g in genes]

    hsl = [e_height + h_height - h * (i + 1) + h / 2 - row_split * len([k for k in range(len(rowbar)) if i >= sum(rowbar[:(k+1)])]) for i in idxes]
    idxes = np.argsort(hsl)
    hsl = [hsl[i] for i in idxes]
    genes = [genes[i] for i in idxes]
    
    pre_hsr = hsl
    pre_hsr_ = hsl
    post_hsr = []
    
    while pre_hsr_ != post_hsr:
        if len(post_hsr) != 0:
            pre_hsr = post_hsr
            pre_hsr_ = post_hsr
            post_hsr = []
            
        while len(pre_hsr) > 0:
            targets = [0]
            center = sum([pre_hsr[i] for i in targets]) / len(targets)
            hs_ = [center + ga_font_size * (len(targets) - 1) / 2 - ga_font_size * j for j in range(len(targets))]
            upper = [j for j in range(len(pre_hsr)) if 0 < pre_hsr[j] - hs_[0] < ga_font_size]
            lower = [j for j in range(len(pre_hsr)) if 0 < hs_[-1] - pre_hsr[j] < ga_font_size]
            
            while len(upper) + len(lower) > 0:
                targets += upper
                targets += lower
                pre_hsr_t = [pre_hsr[i] for i in targets]
                center = sum(pre_hsr_t) / len(targets)
                hs_ = [center + ga_font_size * (len(targets) - 1) / 2 - ga_font_size * j for j in range(len(targets))]
                upper = [j for j in range(len(pre_hsr)) if 0 < pre_hsr[j] - hs_[0] < ga_font_size]
                lower = [j for j in range(len(pre_hsr)) if 0 < hs_[-1] - pre_hsr[j] < ga_font_size]
                            
            pre_hsr = [pre_hsr[i] for i in range(len(pre_hsr)) if pre_hsr[i] not in [pre_hsr[i] for i in targets]]
            post_hsr += hs_[::-1]    
    
    hsr = post_hsr                        
    x = s_width + h_width + col_split * len(colbar)
    for i in range(len(hsl)):
        p.line([x, x+(ga_line_length/3)], [hsl[i], hsl[i]], line_color="black", line_width=ga_line_width)
        p.line([x+(ga_line_length/3),x+2*(ga_line_length/3)], [hsl[i], hsr[i]], line_color="black", line_width=ga_line_width)
        p.line([x+2*(ga_line_length/3),x+ga_line_length], [hsr[i], hsr[i]], line_color="black", line_width=ga_line_width)
        source = dict(x=[x+4*(ga_line_length/3)], y=[hsr[i]], t=[genes[i]])
        p.text(
            x='x', y='y', text='t',
            text_font_size=str(ga_font_size) + "px",
            text_align='left',
            text_baseline='middle',
            text_font_style='italic',
            source=source
        )
                
    
    #################################
    # cluster colors legend
    #################################
    # title
    source = ColumnDataSource(dict(t=["Cluster"]))
    p.text(
        x=cluster_legend_x,
        y=cluster_legend_y,
        text='t',
        text_font_size=str(cluster_title_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )
    # rects
    hs = [cluster_legend_y - (cluster_legend_font_size * 1.8) * (i+1) for i in range(len(nums))]
    c = [color_map[i] for i in nums]
    source = ColumnDataSource(dict(h=hs, c=c))
    p.rect(
        cluster_legend_x,
        'h',
        cluster_legend_font_size * 1.2,
        cluster_legend_font_size * 1.2,
        color='c',
        line_color='c',
        source=source
    )
    # cluster name
    cluster_legend_text_x = cluster_legend_x + cluster_legend_font_size * 1.2
    t = [legend_map[i] for i in nums]
    source = ColumnDataSource(dict(h=hs, t=t))
    p.text(
        x=cluster_legend_text_x,
        y='h',
        text='t',
        text_font_size=str(cluster_legend_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )

    
    #################################
    # tissue colors legend
    #################################
    # title
    source = ColumnDataSource(dict(t=["Tissue"]))
    p.text(
        x=tissue_legend_x,
        y=tissue_legend_y,
        text='t',
        text_font_size=str(tissue_title_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )
    # rects
    hs = [tissue_legend_y - (tissue_legend_font_size * 1.8) * (i+1) for i in range(len(tissue_legend_map))]
    c = tissue_color_map
    source = ColumnDataSource(dict(h=hs, c=c))
    p.rect(
        cluster_legend_x,
        'h',
        cluster_legend_font_size * 1.2,
        cluster_legend_font_size * 1.2,
        color='c',
        line_color='c',
        source=source
    )
    # tissue name
    tissue_legend_text_x = tissue_legend_x + tissue_legend_font_size * 1.2
    t = tissue_legend_map
    source = ColumnDataSource(dict(h=hs, t=t))
    p.text(
        x=tissue_legend_text_x,
        y='h',
        text='t',
        text_font_size=str(tissue_legend_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )

    # save
    p.background_fill_color = None
    p.border_fill_color = None
    export_png(p, filename=filename)
    
    # show
    output_notebook()
    show(p)

In [ ]:
plot_complex_heatmap_all(
    "ALL", "2.1",
    [
        "Bmx","Depp1","Gja4","Gja5","Gkn3","Sox17",
        "Atf3","Cxcl1","Egr1","Fos","Fosb","Jun","Junb","Jund","Nfkbia","Nr4a1","Sgk1",
        "Col4a1","Col4a2","Hlx","Id1","Id3","Igfbp3","Ahr"
        "Cxcl9","Cxcl10","Irf7","Isg15","Lgals9","Ifit3","Oas1a"
        "Angpt2","Apln","Esm1","Mcam","Nid2","Pdgfb","Pgf","Vim",
        "Apoe","St3gal6",
        "Ccl21a","Chst4","Fut7","Clycam1",
        "Ackr1","lcam1","Nr2f2","Sele","Selp","Vcam1","Vwf"
    ], 3,
    f'{Fig1d}.png'
)

# 2. Complex Heatmap (ATAC)

## 2.1. Get top 75 marker regions

In [ ]:
rscript3 = f"""################################
# This is an R script.
# --------------------------------------------------------
# Filename: make_maker_region_list.r
# --------------------------------------------------------
# How to run
# $ Rscript make_maker_region_list.r
# --------------------------------------------------------
# outputs
# - PLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
# - MLN/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
# - PP/filtered_feature_bc_matrix/coordinate_by_monocle.v2.rev.csv
################################
# Library
library(monocle3)
library(ggplot2)
library(dplyr)
library(SeuratWrappers)
library(stringr)
library(Signac)
library(Seurat)
library(GenomicRanges)
library(future)
library(GenomeInfoDb)
library(EnsDb.Mmusculus.v79)
library(patchwork)
set.seed(1234)

# memory
options(future.globals.maxSize = 16000 * 1024^2)

# Functions for marker regions
markerRegions <- function(obj, tissue="PLN"){{
        # swithcing
        Idents(obj) <- "cdef1"

        # rename clusters
        obj <- RenameIdents(obj, '1' = 'Art')
        obj <- RenameIdents(obj, '3' = "CapEC2'")
        obj <- RenameIdents(obj, '4' = 'CapEC2')
        obj <- RenameIdents(obj, "5" = "CapEC1")
        obj <- RenameIdents(obj, '6' = 'CRP')
        obj <- RenameIdents(obj, '7' = 'TrEC')
        obj <- RenameIdents(obj, '8' = 'HEC')
        obj <- RenameIdents(obj, '9' = 'Vn')

        # set plotting order
        levels(obj) <- c(
                "Art",
                "CapEC2'",
                "CapEC2",
                "CapEC1",
                "CRP",
                "TrEC",
                "HEC",
                "Vn"
        )

        # set color
        colors = c(
                "#D0021B",
                "#BA98FF",
                "#FF789B",
                "#50E3C2",
                "#7ED321",
                "#4A90E2",
                "#F5A623",
                "#9013FE",
        )

        # path
        if (tissue == "PLN"){{
                path_marker_obj = "Signac/PLN/markers.atac.v2.1.RData"
                path_top75 = "Signac/PLN/markers.atac.top75.v2.1.csv"
                path_heatmap = "Signac/PLN/markers.atac.heatmap.v2.1.pdf"
        }} else if (tissue == "MLN"){{
                path_marker_obj = "Signac/MLN/markers.atac.v2.1.RData"
                path_top75 = "Signac/MLN/markers.atac.top75.v2.1.csv"
                path_heatmap = "Signac/MLN/markers.atac.heatmap.v2.1.pdf"
        }} else if (tissue == "PP"){{
                path_marker_obj = "Signac/PP/markers.atac.v2.1.RData"
                path_top75 = "Signac/PP/markers.atac.top75.v2.1.csv"
                path_heatmap = "Signac/PP/markers.atac.heatmap.v2.1.pdf"
        }} else {{
                path_marker_obj = "Signac/markers.atac.v2.1.RData"
                path_top75 = "Signac/markers.atac.top75.v2.1.csv"
                path_heatmap = "Signac/markers.atac.heatmap.v2.1.pdf"
        }}

        # marker regions
        DefaultAssay(obj) <- "ATAC"
        obj.markers.atac <- FindAllMarkers(
                object = obj,
                only.pos = FALSE,
                min.pct = 0.05,
                test.use = "LR"
        )

        # save
        save(obj.markers.atac, file=path_marker_obj)
        obj.markers.atac %>%
                dplyr::filter(avg_log2FC >= 0) %>%
                group_by(cluster) %>%
                top_n(75, avg_log2FC) -> top75.posi
        write.csv(
                top75.posi,
                file = path_top75,
                quote = FALSE,
                row.names = FALSE
        )

        # heatmap
        p <- DoHeatmap(
                obj,
                features = top75.posi$gene,
                group.colors = colors,
                label = FALSE
        )
        ggsave(filename = path_heatmap, plot = p, dpi = 100, width = 14, height = 7)
}}

# Top 75 marker regions
load("Seurat/PLN/out/gex.atac.v2.fixed.RData")
load("Seurat/MLN/out/gex.atac.v2.fixed.RData")
load("Seurat/PP/out/gex.atac.v2.fixed.RData")
load("Seurat/out/gex.atac.v2.fixed.RData")
obj <- list(cds1.seurat, cds2.seurat, cds3.seurat, cds.seurat)
tissue <- c("PLN", "MLN", "PP", "ALL")
for (t in 1:4) {{
    markerRegions(obj[[t]], tissue=tissue[t])
}}

#
exportHeatmapData <- function(tissue){{
    if (tissue == "PLN") {{
        assign("obj", get(load("Seurat/PLN/out/gex.atac.v2.fixed.RData")))
        assign("marker", get(load("Signac/PLN/markers.atac.v2.1.RData")))
        path <- "Signac/PLN/heatmap.data.v2.1.csv"
    }} else if (tissue == "MLN") {{
        assign("obj", get(load("Seurat/MLN/out/gex.atac.v2.fixed.RData")))
        assign("marker", get(load("Signac/MLN/markers.atac.v2.1.RData")))
        path <- "Signac/MLN/heatmap.data.v2.1.csv"
    }} else if (tissue == "PP") {{
        assign("obj", get(load("Seurat/PP/out/gex.atac.v2.fixed.RData")))
        assign("marker", get(load("Signac/PP/markers.atac.v2.1.RData")))
        path <- "Signac/PP/heatmap.data.v2.1.csv"
    }} else {{
        assign("obj", get(load("Seurat/out/gex.atac.v2.fixed.RData")))
        assign("marker", get(load("Signac/markers.atac.v2.1.RData")))
        path <- "Signac/heatmap.data.v2.1.csv"
    }}

    # top 75 selection
    marker %>%
    dplyr::filter(avg_log2FC >= 0) %>%
            group_by(cluster) %>%
            top_n(75, avg_log2FC) -> top75.posi

    # re-calc. scaled data
    DefaultAssay(obj) <- "ATAC"
    obj <- NormalizeData(object=obj, normalization.method="LogNormalize", scale.factor = 10000)
    obj <- FindVariableFeatures(object=obj)
    obj <- ScaleData(object=obj, features=top100.posi$gene)

    # export csv
    col_order <- rownames(obj@meta.data[order(obj@meta.data$cdef1),])
    row_order <- subset(x=top75.posi, subset=gene %in% rownames(obj[["ATAC"]]@scale.data))$gene
    mat.atac <- obj[["ATAC"]]@scale.data[row_order, col_order]
    write.csv(mat.atac, path, quote=FALSE)
}}"""

f = open(f'make_maker_region_list.r', 'w')
f.write(rscript3)
f.close()

## 2.2. Draw heatmap

In [ ]:
def plot_complex_heatmap_atac_all(version, ratio, filename):
    # load csv data
    path = "Signac/heatmap.data.v" + version + ".csv"
    df = pd.read_csv(path, index_col=0)
    
    # load cluster definition
    with open("cluster_def_by_monocle_final.v" + version + ".pkl", "rb") as tf:
        cdef = pickle.load(tf)
    
    # cluster numbers
    nums = sorted(list(set(cdef.values())))
    
    # cluster color map
    color_map = {
        1:"#D0021B",
        4:"#BA98FF",
        5:"#FF789B",
        6:"#50E3C2",
        7:"#7ED321",
        8:"#4A90E2",
        9:"#F5A623",
        10:"#9013FE",
        -1: "black",
    }
    legend_map = {
        1:"Art", # Art
        4:"CapEC2'", # CapEC2'
        5:"CapEC2", # CapEC2
        6:"CapEC1", # CapEC1
        7:"CRP", # CRP
        8:"TrEC", # TrEC
        9:"HEC", # HEC
        10:"Vn", # Vn
        -1:"Others", # Others
    }
    
    # tissue color map
    tissue_color_map = [
        "#1EB100",
        "#FEF156",
        "#ED220D"
    ]

    # tissue legend map
    tissue_legend_map = [
        "PLN",
        "MLN",
        "PP"
    ]

    # column rename
    cols = []
    for c in df.columns.tolist():
        if c.startswith("pln_"):
            cols.append(c.split("_")[1]+"_1")
        elif c.startswith("mln_"):
            cols.append(c.split("_")[1]+"_2")
        else:
            cols.append(c.split("_")[1]+"_3")
            
    # column reorder, downsampling and split info.
    cols_ = []
    colbar = []
    subcolbar = []
    for n in nums:
        subset = [i for i in cols if cdef[i] == n]
        subset_1 = [i for i in subset if i[-1] == "1"]
        subset_2 = [i for i in subset if i[-1] == "2"]
        subset_3 = [i for i in subset if i[-1] == "3"]
        cols_ += subset_1 + subset_2 + subset_3
        
    cols_ = [cols_[i] for i in range(len(cols_)) if i % ratio == 0]
    
    for n in nums:
        subset = [i for i in cols_ if cdef[i] == n]
        subset_1 = [i for i in subset if i[-1] == "1"]
        subset_2 = [i for i in subset if i[-1] == "2"]
        subset_3 = [i for i in subset if i[-1] == "3"]
        colbar.append(len(subset))
        subcolbar.append([len(subset_1), len(subset_2), len(subset_3)])

    cols_ = ["pln_"+i.split("_")[0] if i[-1]=="1" else "mln_"+i.split("_")[0] if i[-1]=="2" else "pp_"+i.split("_")[0] for i in cols_]
    df = df.loc[:, cols_]
            
    
    # row split
    rowbar = [100] * (len(nums))
    
    # custom color scale
    v = list(range(1, 256))[::-1]
    p_to_b = [(i, 0, i) for i in v]
    v = list(range(1, 256))
    b_to_y = [(i, i, 0) for i in v]
    custom_palette = [rgb_to_hex(i) for i in p_to_b + [(0, 0, 0)] + b_to_y]
    

    #################################
    # Parameters
    #################################
    f_width = 1500
    f_height = 900
    h_width = 1000
    h_height = 700
    s_width = 50
    s_height = 100
    e_width = 200
    e_height = 100
    w = h_width / len(df.T)
    h = h_height / len(df)
    q = 1
    
    col_split = q * 4
    row_split = 4
    
    color_max = 2.2
    color_min = -2.2 
    color_bar_x = s_width + h_width + q * 100
    color_bar_y = e_height - row_split * len(rowbar)
    color_bar_w = q * 35
    color_bar_h = 175
    color_bar_ticks_ratio = 6
    color_bar_ticks_line_width_ratio = 75
    color_bar_texts_font_size_ratio = 12
    
    rowbar_x = s_width - q * 17.5
    rowbar_w = q * 25
    
    colbar_y = e_height + h_height + 42.5
    colbar_h = 25

    subcolbar_y = e_height + h_height + 17.5
    subcolbar_h = 25
    
    cluster_title_font_size = 25
    cluster_legend_font_size = 22
    cluster_legend_x = s_width + h_width + q * 100
    cluster_legend_y = e_height + h_height + 50

    tissue_title_font_size = 25
    tissue_legend_font_size = 22
    tissue_legend_x = s_width + h_width + q * 100
    tissue_legend_y = e_height + h_height + 50 - 462
    
    
    #################################
    # Figure
    #################################
    p = figure(tools="save", width=f_width, height=f_height)
    p.grid.grid_line_color = None
    p.axis.axis_line_color = None
    p.axis.major_tick_line_color = None
    p.x_range = Range1d(0, f_width)
    p.y_range = Range1d(0, f_height)
    

    #################################
    # Heatmap
    #################################
    x = []
    y = []
    val = []
    for i in range(len(df.T)):
        for j in range(len(df)):
            cs = col_split * len([k for k in range(len(colbar)) if i >= sum(colbar[:(k+1)])]) # column split
            rs = row_split * len([k for k in range(len(rowbar)) if j >= sum(rowbar[:(k+1)])]) # row split
            x.append(s_width + w * i + cs + w / 2)
            y.append(h_height + e_height - h * (j + 1) - rs + h / 2)    
            val.append(df.iloc[j, i])

    source = ColumnDataSource(dict(x=x, y=y, val=val))
    linear_transform_color = linear_cmap(field_name="val", palette=custom_palette, low=color_min, high=color_max)
    p.rect('x', 'y', w, h, source=source, color=linear_transform_color, line_color=linear_transform_color, line_width=1)


    #################################
    # Color bar
    #################################
    # color rects
    x = []
    y = []
    col = []
    hu = color_bar_h / len(custom_palette)
    for i in range(len(custom_palette)):
        x.append(color_bar_x)
        y.append(color_bar_y + i * hu + hu / 2)
        col.append(custom_palette[i])

    source = ColumnDataSource(dict(x=x, y=y, col=col))
    p.rect('x', 'y', color_bar_w, hu, source=source, color='col', line_color='col')
    
    # ticks
    high = math.floor(color_max)
    low = math.ceil(color_min)
    step = (high - low) // 5 if high - low >= 5 else 1
    steps = list(range(low, high + 1, step))
    xrs = color_bar_x + (color_bar_w / 2) - (color_bar_w / color_bar_ticks_ratio)
    xre = color_bar_x + (color_bar_w / 2)
    xls = color_bar_x - (color_bar_w / 2) + (color_bar_w / color_bar_ticks_ratio)
    xle = color_bar_x - (color_bar_w / 2)
    y = [color_bar_y + (i - color_min) *(color_bar_h / (color_max - color_min)) for i in steps]
    lw = color_bar_h / color_bar_ticks_line_width_ratio
    for y_ in y:
        p.line([xrs, xre], [y_, y_], line_color = "white", line_width = lw)
        p.line([xls, xle], [y_, y_], line_color = "white", line_width = lw)

    # texts
    x = [color_bar_x + (color_bar_w / 2) + (color_bar_w / color_bar_ticks_ratio)] * len(y)
    t = [str(s) for s in steps]
    source = ColumnDataSource(dict(x=x, y=y, t=t))
    font_size = str(round(color_bar_h / color_bar_texts_font_size_ratio))+"px"
    p.text(x='x', y='y', text='t', text_font_size=font_size, text_align="left", text_baseline="middle", source=source)
    
    
    #################################
    # row colors
    #################################
    y = [h_height + e_height - row_split * i - h * sum(rowbar[:i]) - h * (rowbar[i] / 2) for i in range(len(rowbar))]
    hs = [h * i for i in rowbar]
    c = [color_map[nums[i]] for i in range(len(rowbar))]
    source = ColumnDataSource(dict(y=y, h=hs, c=c))
    p.rect(rowbar_x, 'y', rowbar_w, 'h', color='c', line_color='c', source=source)

    
    #################################
    # col colors
    #################################
    x = [s_width + col_split * i + w * sum(colbar[:i]) + w * (colbar[i] / 2) for i in range(len(colbar))]
    ws = [w * i for i in colbar]
    c = [color_map[nums[i]] for i in range(len(colbar))]
    source = ColumnDataSource(dict(x=x, w=ws, c=c))
    p.rect('x' , colbar_y, 'w', colbar_h, color='c', line_color='c', source=source)


    #################################
    # subcol colors
    #################################
    x = []
    ws = []
    c = []
    for i in range(len(subcolbar)):
        for j in range(len(subcolbar[i])):
            x.append(s_width + col_split * i + w * sum(colbar[:i]) + w * sum(subcolbar[i][:j]) + w * (subcolbar[i][j] / 2))
            ws.append(w * subcolbar[i][j])
            c.append(tissue_color_map[j])
    source = ColumnDataSource(dict(x=x, w=ws, c=c))
    p.rect('x' , subcolbar_y, 'w', subcolbar_h, color='c', line_color='c', source=source)
                    
    
    #################################
    # cluster colors legend
    #################################
    # title
    source = ColumnDataSource(dict(t=["Cluster"]))
    p.text(
        x=cluster_legend_x,
        y=cluster_legend_y,
        text='t',
        text_font_size=str(cluster_title_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )
    # rects
    hs = [cluster_legend_y - (cluster_legend_font_size * 1.8) * (i+1) for i in range(len(nums))]
    c = [color_map[i] for i in nums]
    source = ColumnDataSource(dict(h=hs, c=c))
    p.rect(
        cluster_legend_x,
        'h',
        cluster_legend_font_size * 1.2,
        cluster_legend_font_size * 1.2,
        color='c',
        line_color='c',
        source=source
    )
    # cluster name
    cluster_legend_text_x = cluster_legend_x + cluster_legend_font_size * 1.2
    t = [legend_map[i] for i in nums]
    source = ColumnDataSource(dict(h=hs, t=t))
    p.text(
        x=cluster_legend_text_x,
        y='h',
        text='t',
        text_font_size=str(cluster_legend_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )

    
    #################################
    # tissue colors legend
    #################################
    # title
    source = ColumnDataSource(dict(t=["Tissue"]))
    p.text(
        x=tissue_legend_x,
        y=tissue_legend_y,
        text='t',
        text_font_size=str(tissue_title_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )
    # rects
    hs = [tissue_legend_y - (tissue_legend_font_size * 1.8) * (i+1) for i in range(len(tissue_legend_map))]
    c = tissue_color_map
    source = ColumnDataSource(dict(h=hs, c=c))
    p.rect(
        cluster_legend_x,
        'h',
        cluster_legend_font_size * 1.2,
        cluster_legend_font_size * 1.2,
        color='c',
        line_color='c',
        source=source
    )
    # tissue name
    tissue_legend_text_x = tissue_legend_x + tissue_legend_font_size * 1.2
    t = tissue_legend_map
    source = ColumnDataSource(dict(h=hs, t=t))
    p.text(
        x=tissue_legend_text_x,
        y='h',
        text='t',
        text_font_size=str(tissue_legend_font_size) + "px",
        text_align="left",
        text_baseline="middle",
        source=source
    )

    # save
    p.background_fill_color = None
    p.border_fill_color = None
    export_png(p, filename=filename)
    
    # show
    output_notebook()
    show(p)

In [ ]:
plot_complex_heatmap_atac_all("2.1", 2, f'{Supplementary_Fig2}.png')